In [1]:
from google import genai
import enum
import json
from time import sleep

import utils
from importlib import reload

In [30]:
reload(utils)

<module 'utils' from '/work/pi_brenocon_umass_edu/marisa/latin-qa/structured_output_scripts/utils.py'>

In [2]:
with open('../ocr/gemini/gemini-key.txt', 'r') as f:
    api_key = f.read().strip()
client = genai.Client(api_key=api_key)

In [3]:
with open('nle_prompt.txt', 'r') as f:
    PROMPT_TEMPLATE = f.read()
with open('nle_prompt_reading_comp.txt', 'r') as f:
    PROMPT_TEMPLATE_READING_COMP = f.read()


In [4]:
year=2020
source_name = "NLE"

In [5]:
exam_filename = f'../data/semi_structured/NLE_{year}/exams_{year}.json'
key_filename = f'../data/semi_structured/NLE_{year}/answer_keys_{year}.json'

with open(exam_filename, 'r') as f:
    exams = json.load(f)

with open(key_filename, 'r') as f:
    keys = json.load(f)



In [6]:
def exam_to_key_name(exam_name):
    exam_name = exam_name.replace('**', '')

    # if "LATIN" not in exam_name, need to 
    # add it after the level name (the first word)
    if "LATIN" not in exam_name:
        split_name = exam_name.split(' ')
        exam_name = split_name[0] + ' LATIN ' + ' '.join(split_name[1:])
    
    return exam_name + " EXAM"


In [7]:
# for exams before 2021
def key_to_exam_name(key_name):
    return key_name.upper()

In [8]:
def get_difficulty(exam_name):
    if exam_name.startswith("INTRODUCTION"):
        return "beginner"
    elif "BEGINNER" in exam_name or "BEGINNING" in exam_name:
        return "beginner"
    elif "INTERMEDIATE" in exam_name:
        return "intermediate"
    elif "ADVANCED" in exam_name:
        return "advanced"
    elif "V" in exam_name: # III-IV and above is advanced
        return "advanced"
    elif "II" in exam_name: # will match II or III 
        return "intermediate"
    elif "I" in exam_name: # will match I only
        return "beginner"
    else:
        return "unknown"

In [9]:
def get_passages(exam):
    passages = []

    for q in exam:
        if q.startswith("passage_questions"):
            n = int(q.split('_')[-1])

            passages.append({
                "text": exam["passage_" + str(n)],
                "questions": exam[q]
            })

    return passages


In [10]:
for exam in exams:
    print(exam)


LATIN I
LATIN II
LATIN III
LATIN III-IV POETRY
LATIN III-IV PROSE
INTRODUCTION TO LATIN
LATIN V-VI


In [11]:
for key_ in keys:
    print(key_)

Introduction to Latin
Latin I
Latin II
Latin III
Latin III-IV Prose
Latin III-IV Poetry
Latin V-VI


In [14]:
def process_reading_comp_question(exam_name, key_name):
    
    exam = exams[exam_name]
    key = keys[key_name]


    # get passages
    passages = get_passages(exam)

    save_passages = []
    save_questions = []

    # process reading comp questions first
    for p, passage_dict in enumerate(passages):
        text = passage_dict["text"]
        question_nums = passage_dict["questions"]
        
        # generate a unique passage id
        exam_underscored = exam_name.replace(' ', '_')
        p_id = f"{source_name}_{year}_{exam_underscored}_{p}"
        
        # make Passage object
        passage = utils.Passage(
            passage_id=p_id,
            text=text,
            language=utils.Language.LATIN,
            source_name=source_name,
            source_year=year,
            questions=question_nums
            )
        passage_json = passage.model_dump_json()
        passage_json = passage_json.replace('null', 'None')
        save_passages.append(eval(passage_json))
        
        for n in question_nums:
            q_id = f"{source_name}_{year}_{exam_underscored}_{n}"
            question = exam[n]
            question_text = question['question']
            answer_choices = {k:v for (k,v) in question.items() if k != 'question'}

            correct_answer_letter = key[n]
            correct_answer = answer_choices[correct_answer_letter]

            answer_choices_strs = [f"{k}: {v}" for (k,v) in answer_choices.items()]
            correct_answer_str = f"{correct_answer_letter}: {correct_answer}"

            prompt = PROMPT_TEMPLATE_READING_COMP.format(
                source_name,
                year,
                get_difficulty(exam_name),
                q_id,
                p_id,
                question_text,
                answer_choices_strs,
                correct_answer_str
            )
            
            response = client.models.generate_content(
                model='gemini-2.0-flash',
                contents=prompt,
                config=utils.config
            )
            print(response)
            response_text = response.text
            response_text = response_text.replace('null', 'None')
            save_questions += eval(response_text)
            sleep(0.2)

    return save_questions, save_passages
    

In [12]:
if year > 2020:
    exam_key_pairs = [(exam_name, exam_to_key_name(exam_name)) for exam_name in exams]
else:
    exam_key_pairs = [(key_to_exam_name(key_name), key_name) for key_name in keys]

exam_key_pairs


[('INTRODUCTION TO LATIN', 'Introduction to Latin'),
 ('LATIN I', 'Latin I'),
 ('LATIN II', 'Latin II'),
 ('LATIN III', 'Latin III'),
 ('LATIN III-IV PROSE', 'Latin III-IV Prose'),
 ('LATIN III-IV POETRY', 'Latin III-IV Poetry'),
 ('LATIN V-VI', 'Latin V-VI')]

In [15]:

all_questions = []
all_passages = []
for (exam_name, key_name) in exam_key_pairs:
    print(exam_name)
    questions, passages = process_reading_comp_question(exam_name, key_name)
    all_questions += questions
    all_passages += passages
    
    # save intermediate results
    with open(f'../data/structured/nle_reading_comp_questions_{year}.json', 'w') as f:
        json.dump(all_questions, f, indent=4)
    with open(f'../data/structured/nle_reading_comp_passages_{year}.json', 'w') as f:
        json.dump(all_passages, f, indent=4)


     

INTRODUCTION TO LATIN
Organize the following questions and answers into a structured json output. I will provide the values for some fields, but you must fill out the remaining fields. 
Do not assign a difficulty level yourself. If the given difficulty is unknown, assign "unknown".
Do not edit any fields that I provide.
If a question is asked in English, you should assign the question_language as English. This includes who, what, when, where, why, and how questions. This also includes instructions, such as translate, give a synonym, etc. Only assign Latin if the question itself is asked in Latin, not only if Latin words are used in the question.
If the answer is a non-English name, assign the answer_language as Latin. Try to assign the language used in the majority of the answer, or in the most important part of the answer. Only label the language as "both" if the question specifically asked for an answer in both languages.
To determine the correctness logic, first determine if there a

In [65]:
def process_other_questions(exam_name, key_name):
    exam = exams[exam_name]
    key_ = keys[key_name]
    exam_underscored = exam_name.replace(' ', '_')

    # get passages
    passages = get_passages(exam)
    #print(passages)
    all_passage_questions = []
    for passage_dict in passages:
        question_nums = passage_dict["questions"]
        all_passage_questions += question_nums

    all_passage_questions = set(all_passage_questions)
    all_questions = set(list(key_.keys()))
    other_questions = all_questions - all_passage_questions

    #print('passage questions:', all_passage_questions)
    #print('other_questions:', other_questions)

    save_questions = []
    
    # process questions
    for q in other_questions:
        q_id = f"{source_name}_{year}_{exam_underscored}_{q}"
        
        if q not in exam:
            continue
        
        question = exam[q]
        question_text = question['question']
        answer_choices = {k:v for (k,v) in question.items() if k != 'question'}
        correct_answer_letter = key_[q]
        correct_answer = answer_choices[correct_answer_letter]

        answer_choices_strs = [f"{k}: {v}" for (k,v) in answer_choices.items()]
        correct_answer_str = f"{correct_answer_letter}: {correct_answer}"

        prompt = PROMPT_TEMPLATE.format(
            source_name,
            year,
            get_difficulty(exam_name),
            q_id,
            question_text,
            answer_choices_strs,
            correct_answer_str
        )

        response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompt,
            config=utils.config
        )
        print(response)
        response_text = response.text
        response_text = response_text.replace('null', 'None')
        save_questions += eval(response_text)
        sleep(0.2)

    return save_questions
    

In [ ]:
other_questions = []
for exam_name in exams:
    print(exam_name)
    key_name = exam_to_key_name(exam_name)
    other_questions += process_other_questions(exam_name, key_name)

    # save intermediate results
    with open(f'../data/structured/nle_other_questions_{year}.json', 'w') as f:
        json.dump(other_questions, f, indent=4)
    

    